# Program for training simple model for back-classification

Create kernel: `uv run ipython kernel install --user --name="python-classification"`
Switch to this kernel (see top right in jupyter)

Also ran one time: `mlflow experiments create --experiment-name "back_classification" --artifact-location s3://projet-aiml4os-wp10/Cluster5/mlflow_artifacts`. Not sure if I need to now as have changed things around

The program is divided into the following general processes:

- [Initial setup](#setup)
- [Read in data](#read)
- [Filter data](#filter)
- [Pre-process text variables](#preprocess)
- [Train model](#train)

<a id="setup"></a>
## Initial Setup
Here we fetch the necessary Python packages for the program. This also includes fetching additional functions from the utils folder and variables from the configuration file (config).

In [ ]:
import sys
print(sys.executable)

In [ ]:
import pandas as pd
import os

from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.calibration import CalibratedClassifierCV

import nltk
from nltk.corpus import stopwords

from utils.one_to_one import get_1to1_dict, get_problem_NACE
from utils.rules_classification import norwegian_rules
from utils.text_processing import preprocess
from config import path_info, bucket_info, model_hyperparameter

import mlflow
import mlflow.sklearn

In [ ]:
# Location for model and data
bucket_url = bucket_info["bucket_url"]
bucket = "projet-aiml4os-wp10/Cluster5"

train_path = f"{bucket_url}/train_doublenace_2026-03-23.parquet"
conversion_path = f"{bucket_url}/NACE2.1-NACE2_Table_V1.05.xlsx"

In [ ]:
os.environ["S3_ENDPOINT"] = "https://minio.lab.sspcloud.fr"
os.environ["AWS_S3_ENDPOINT"] = "https://minio.lab.sspcloud.fr"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.environ["S3_ENDPOINT"]

In [ ]:
os.environ["AWS_DEFAULT_REGION"] = 'us-east-1'

In [ ]:
os.getenv('AWS_S3_ENDPOINT')

In [ ]:
from getpass import getpass

os.environ["AWS_ACCESS_KEY_ID"] = getpass("AWS_ACCESS_KEY_ID: ")
os.environ["AWS_SECRET_ACCESS_KEY"] = getpass("AWS_SECRET_ACCESS_KEY: ")
os.environ["AWS_SESSION_TOKEN"] = getpass("AWS_SESSION_TOKEN: ")


In [ ]:
# In bash, $ export xxx = yyy  - but doesn't import to correct env

In [ ]:
# Original experiment setup # 

In [ ]:
# Set MLflow experiment for ML model if not set from before

if not mlflow.get_experiment_by_name("back_classification"):
    mlflow.create_experiment(
        "back_classification",
        artifact_location=f"s3://{bucket}/mlflow_artifacts/"
    )

mlflow.set_experiment("back_classification")

<a id="read"></a>
## Read in data

In [ ]:
# Read in training data
train = pd.read_parquet(path_info["train_path"])
print(f'Number of observations in training: {train.shape[0]}')

<a id="filter"></a>
## Filter data
**Remove 1-to-1 groups**

In [ ]:
one_to_one_dict = get_1to1_dict(path_info["conversion_path"])
mask_1 = train.nace_21_code.isin(one_to_one_dict.keys())
print(f'Number of observations in one-to-one nace: {mask_1.sum()}')

train = train.loc[~mask_1, :].copy()

**Remove problem groups**

In [ ]:
problem_nace = get_problem_NACE(path_info["conversion_path"])
mask_problem = train.nace_21_code.isin(problem_nace)
print(f'Number of observations in problem nace: {mask_problem.sum()}')

train = train.loc[~mask_problem, :].copy()

**Remove rules groups**

In [ ]:
mask_rules = train.nace_21_code.isin(norwegian_rules.keys())
print(f'Number of observations in rules nace: {mask_rules.sum()}')

train = train.loc[~mask_rules, :].copy()

**Remove groups with too small counts (<3)**

In [ ]:
# Check minimum number
tab = train.groupby("nace_20_code").orgnr.count()
print(f'Minium number of observations in NACE class: {min(tab)}')

In [ ]:
small_groups = tab[tab<3].index
mask_small = train.nace_20_code.isin(small_groups.tolist())
print(f"The following NACE rev. 2.0 have less than 3 observations and have been removed: {small_groups.tolist()}")
print(f"Number of observations removed due to less than 3 observations: {mask_small.sum()}")

train = train.loc[~mask_small,:].copy()

<a id="preprocess"></a>
## Preprocess text

In [ ]:
# Get stopwords
nltk.download('stopwords') # one time to use locally
norwegian_stopwords = set(stopwords.words('norwegian'))|{"as"}

In [ ]:
# Create text variable
train["company_name"] = train["company_name"].fillna("")
train["company_activity"] = train["company_activity"].fillna("")
train["company_text"] = train["company_name"] + " " + train["company_activity"]

In [ ]:
# Preprocess and add in nace 2.1
train["company_text_processed"] = preprocess(train, 
                                             text_variable="company_text", 
                                             stopwords = norwegian_stopwords, 
                                             nace_21_variable = "nace_21_code"
                                            )

<a id="train"></a>
## Train model


In [ ]:
# using small set for testing
mask_little = train.nace_20_code.isin(["18.12","25.61","28.21","31.02"])
train = train.loc[mask_little,:].copy()
train.groupby("nace_20_code").count()

In [ ]:
with mlflow.start_run() as run:
    pipe_svc = Pipeline([('countvec', CountVectorizer(max_features=model_hyperparameter['max_features'])),
                             ('tf', TfidfTransformer(use_idf=True)),
                             ('clf', CalibratedClassifierCV(estimator=LinearSVC(C=model_hyperparameter['C']),
                                                            cv=3, 
                                                            method='sigmoid',
                                                            n_jobs=5),
                             )
                            ])
    pipe_svc.fit(train.company_text_processed , train.nace_20_code)

    result = mlflow.sklearn.log_model(sk_model=pipe_svc, name="model")
    print("Model URI:", result.model_uri)

In [ ]:
pipe_svc

### MISC CODE

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

print("MLflow version:", mlflow.__version__)

experiments = client.search_experiments(
    view_type=mlflow.entities.ViewType.ALL
)

for exp in experiments:
    if exp.name == "back_classification":
        print(
            "ID:", exp.experiment_id,
            "| Status:", exp.lifecycle_stage,
            "| Artifact location:", exp.artifact_location
        )

In [ ]:
client.restore_experiment("1")

# Rename it to free up the original name
client.rename_experiment(
    "1",
    "back_classification_old"
)


In [ ]:
# Fixes, rename old experiment and start with fresh ID

In [ ]:

if not mlflow.get_experiment_by_name("back_classification"):
    new_id = client.create_experiment(
        "back_classification",
        artifact_location=f"s3://{bucket}/mlflow_artifacts/"
    )

mlflow.set_experiment("back_classification")

print(new_id)
print(client.get_experiment(new_id).artifact_location)